# 01 - Exploratory data analysis (EMBER2024, Win64)

This notebook is a narrative view only. Every number comes from `src.data` and from the files that
`run_all.py --steps data` wrote. No logic lives here.

**We only look at the `train` split.** `test_random` and `test_future` stay unseen until final
scoring, so nothing we notice here can steer modelling choices.

In [ ]:
# Make the project root importable when the notebook is opened from notebooks/.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src.data import feature_groups, load_split

X, y, meta = load_split("train")
print(f"train: X {X.shape} {X.dtype}, quick={config.QUICK}")

## Data checks from `verify_data.py`

`scripts/verify_data.py` writes every count and flag to `reports/data_check.json`. The flags are
shown here so nobody has to open the file. Expect a flag about `detection_ratio`, which is almost a
copy of the label.

In [ ]:
check_path = config.REPORTS / f"data_check{config.RUN_SUFFIX}.json"
report = json.loads(check_path.read_text(encoding="utf-8"))
print("hard checks:", report.get("hard_checks"))
for flag in report.get("flags", []):
    print("FLAG:", flag)

## Class balance

EMBER2024 is exactly 50/50 benign/malicious every week **by construction**. Real traffic is mostly
benign, so precision measured here is optimistic. This goes in the bias section of the report.

In [ ]:
meta["label"].map({0: "benign", 1: "malicious"}).value_counts().to_frame("count")

## Samples per week

`train` is a stratified (label x week) sample of the 52 training weeks, so each week should keep its
share and its 50/50 balance.

In [ ]:
per_week = meta.groupby(["week", "label"]).size().unstack("label").rename(columns={0: "benign", 1: "malicious"})
ax = per_week.plot(kind="bar", stacked=True, figsize=(12, 3), width=0.9)
ax.set_xlabel("week (0 = 2023-09-24)")
ax.set_ylabel("samples in train")
ax.set_xticks(ax.get_xticks()[::4])
plt.tight_layout()
per_week.describe()

## Feature families

The 2,568 columns come in 12 families. The column ranges come from `feature_groups()`, which reads
them from the thrember extractor. Below, each sample's mean value inside a few families is compared
between benign and malicious files. This is a rough look at whether a family separates the classes.

In [ ]:
groups = pd.DataFrame(feature_groups(), columns=["family", "start", "end"])
groups["width"] = groups["end"] - groups["start"]
groups

In [ ]:
families = ["general", "byteentropy", "strings", "section", "imports", "authenticode"]
ranges = groups.set_index("family").loc[families, ["start", "end"]]
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (name, start, end) in zip(axes.flat, ranges.itertuples()):
    # log1p of the per-sample mean, because several families span many orders of magnitude
    family_mean = np.log1p(np.abs(X[:, start:end]).mean(axis=1))
    for label, colour in ((0, "tab:blue"), (1, "tab:orange")):
        ax.hist(family_mean[y == label], bins=60, alpha=0.6, color=colour,
                label="benign" if label == 0 else "malicious")
    ax.set_title(f"{name} [{start}, {end})")
axes.flat[0].legend()
plt.tight_layout()

## A time proxy inside the standard features

Column 696 is `header.coff.timestamp`, the PE compile time. It is part of the normal 2,568-dim
vector, so models can learn *when* a file was built. That helps more on a random split (train and
test come from the same months) than on future weeks. Expect it to rank high in importance.

In [ ]:
compile_time = pd.to_datetime(X[:, 696].astype("int64"), unit="s", errors="coerce")
pd.DataFrame({"label": y, "compile_year": compile_time.year}).groupby("label")["compile_year"].describe()

## `detection_ratio` vs label: the leaky feature

`detection_ratio` is the share of antivirus engines that flag the file. The label is derived from
those same verdicts, so this column is an almost perfect label proxy. It is **never** part of `X` in
the main runs. Step 4 adds it on purpose to see which tools notice the leak.

In [ ]:
positives = meta["detection_ratio"].str.split("/").str[0].astype(int)
bins = pd.cut(positives, bins=[-1, 0, 4, 10, 30, 100], labels=["0", "1-4", "5-10", "11-30", ">30"])
pd.crosstab(meta["label"].map({0: "benign", 1: "malicious"}), bins)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for label, colour in ((0, "tab:blue"), (1, "tab:orange")):
    ax.hist(meta.loc[meta["label"] == label, "detection_ratio_float"], bins=50, alpha=0.6, color=colour,
            label="benign" if label == 0 else "malicious")
ax.set_xlabel("detection_ratio (share of engines flagging the file)")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()